In [ ]:
from PIL import Image
import cv2
import numpy as np
from google.colab.patches import cv2_imshow

In [ ]:
import zipfile

path = '/content/drive/MyDrive/Visão Computacional Guia Completo/Datasets/yalefaces.zip'
zip_objetic = zipfile.ZipFile(file=path, mode='r')
zip_objetic.extractall('./')
zip_objetic.close

<bound method ZipFile.close of <zipfile.ZipFile filename='/content/drive/MyDrive/Visão Computacional Guia Completo/Datasets/yalefaces.zip' mode='r'>>

In [ ]:
import os
print(os.listdir('/content/yalefaces/train'))

['subject15.surprised.gif', 'subject02.surprised.gif', 'subject07.glasses.gif', 'subject13.happy.gif', 'subject13.wink.gif', 'subject09.glasses.gif', 'subject07.noglasses.gif', 'subject13.centerlight.gif', 'subject03.normal.gif', 'subject02.noglasses.gif', 'subject09.happy.gif', 'subject12.sad.gif', 'subject14.glasses.gif', 'subject01.noglasses.gif', 'subject03.noglasses.gif', 'subject06.sad.gif', 'subject07.sleepy.gif', 'subject08.sad.gif', 'subject04.sleepy.gif', 'subject12.noglasses.gif', 'subject02.sleepy.gif', 'subject14.centerlight.gif', 'subject14.leftlight.gif', 'subject10.wink.gif', 'subject02.glasses.gif', 'subject07.sad.gif', 'subject09.sleepy.gif', 'subject11.surprised.gif', 'subject12.wink.gif', 'subject14.rightlight.gif', 'subject09.surprised.gif', 'subject02.normal.gif', 'subject05.glasses.gif', 'subject04.glasses.gif', 'subject10.sleepy.gif', 'subject07.centerlight.gif', 'subject05.rightlight.gif', 'subject03.surprised.gif', 'subject12.sleepy.gif', 'subject08.surprised.

In [ ]:
def get_image_data():
  paths = [os.path.join('/content/yalefaces/train', f) for f in os.listdir('/content/yalefaces/train')]
  faces = []
  ids = []
  for path in paths:
    imagem = Image.open(path).convert('L')
    imagem_np = np.array(imagem, 'uint8')
    id = int(os.path.split(path)[1].split('.')[0].replace('subject', ''))
    ids.append(id)
    faces.append(imagem_np)

  return np.array(ids), faces

In [ ]:
ids, faces = get_image_data()

In [ ]:
lbph_classicador = cv2.face.LBPHFaceRecognizer_create(radius=4, neighbors=14, grid_x=9, grid_y=9)
lbph_classicador.train(faces, ids)
lbph_classicador.write('lbph_classificier.yml')

In [ ]:
model = cv2.face.LBPHFaceRecognizer_create()
model.read('/content/lbph_classificier.yml')

In [ ]:
imagem_teste = '/content/yalefaces/test/subject10.sad.gif'

In [ ]:
imagem = Image.open(imagem_teste).convert('L')
imagem_np = np.array(imagem, 'uint8')

In [ ]:
previsão = model.predict(imagem_np)
previsão

(10, 36.84750735484425)

In [ ]:
saida_esperada = int(os.path.split(imagem_teste)[1].split('.')[0].replace('subject', ''))
saida_esperada

10

In [ ]:
cv2.putText(imagem_np, 'Pred:' + str(previsão[0]), (10, 30), cv2.FONT_HERSHEY_COMPLEX_SMALL, 1, (0, 255, 0))
cv2.putText(imagem_np, 'Saida:' + str(previsão[0]), (10, 50), cv2.FONT_HERSHEY_COMPLEX_SMALL, 1, (0, 255, 0))
cv2_imshow(imagem_np)

In [ ]:
paths =[os.path.join('/content/yalefaces/test',f) for f in os.listdir('/content/yalefaces/test')]
previsoes = []
saidas_esperadas = []
for path in paths:
  imagem  = Image.open(path).convert('L')
  imagem_np = np.array(imagem, 'uint8')
  previsao, _ = model.predict(imagem_np)
  saida_esperada = int(os.path.split(path)[1].split(".")[0].replace('subject', ''))
  previsoes.append(previsao)
  saidas_esperadas.append(saida_esperada)

In [ ]:
previsoes = np.array(previsoes)
saidas_esperadas = np.array(saidas_esperadas)

In [ ]:
previsoes

array([11,  4,  9, 11, 12, 12,  3,  4,  4,  3, 12,  5,  7,  4,  8, 13,  1,
        4,  5, 10,  6, 15,  4,  9, 14,  7, 13,  7, 14, 14])

In [ ]:
saidas_esperadas

array([11,  4,  9, 11, 12,  2,  3,  4,  8,  3, 12,  5,  9, 10,  8, 13,  1,
        1,  5, 10,  6, 15,  2,  7, 14,  7, 13, 15,  6, 14])

In [ ]:
from sklearn.metrics import accuracy_score
accuracy_score(saidas_esperadas, previsoes)

0.7

In [ ]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(saidas_esperadas, previsoes)
cm

array([[1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0],
       [0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0],
       [0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0],
       [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1]])

In [ ]:
import seaborn as sns
sns.heatmap(cm, annot=True);